# DoorDash Delivery ETA — Etapa 06: Treinamento e Avaliação (Google Colab Autocontido)

Este notebook é **100% autocontido** para execução no **Google Colab**, dependendo exclusivamente do arquivo `historical_data.csv` (sem necessidade de enviar `src/`, relatórios ou notebooks anteriores).

Ele reproduz fielmente todas as decisões metodológicas das Etapas 01 a 05:
1. **Preparação controlada (Etapa 02):** remoção apenas dos 7 registros sem `actual_delivery_time`; conversão de negativos em `NaN` apenas nas 4 colunas especificadas; sem imputação global e sem remoção de outliers.
2. **Engenharia de atributos (Etapas 04 e 05):** atributos temporais no horário registrado, atributos do pedido (incluindo `invalid_item_price_range`), `historical_store_order_count` com tratamento de empates `(store_id, created_at)` e separação rigorosa entre atributos **confirmados**, **experimentais** e **proibidos**.
3. **Validação temporal com embargo (Etapa 05):** cortes em `2015-02-10 10:00:00` e `2015-02-14 10:00:00`, garantindo `138.572` (treino), `28.808` (validação) e `30.041` (teste isolado).
4. **Treinamento e ablação (Etapa 06):** Baseline (mediana), Linear Regression, Random Forest e HistGradientBoosting no conjunto conservador, seguidos de ablações separadas para atributos de disponibilidade incerta.

## 1. Preparação do Ambiente e Carregamento de `historical_data.csv`

In [ ]:
# Verificação de dependências essenciais (já pré-instaladas no Google Colab)
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

In [ ]:
# Escolha o modo de carregamento:
# - "upload": abre janela de upload direto no Google Colab (padrão)
# - "drive": carrega a partir do Google Drive montado
# - "local": carrega de caminho local (caso rode fora do Colab)
LOAD_MODE = "upload"
DRIVE_CSV_PATH = "/content/drive/MyDrive/datasets/historical_data.csv"
LOCAL_CSV_PATH = "historical_data.csv"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and LOAD_MODE == "upload":
    from google.colab import files
    print("Selecione o arquivo historical_data.csv para upload:")
    uploaded = files.upload()
    csv_path = next(iter(uploaded.keys()))
elif IN_COLAB and LOAD_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    csv_path = DRIVE_CSV_PATH
else:
    csv_path = LOCAL_CSV_PATH

assert Path(csv_path).exists(), f"Arquivo não encontrado: {csv_path}"
df_raw = pd.read_csv(csv_path)
print(f"Arquivo carregado com sucesso: {csv_path} | Shape: {df_raw.shape}")

## 2. Preparação Controlada dos Dados (Etapa 02)

In [ ]:
NEGATIVE_TO_NAN_COLS = (
    "min_item_price",
    "total_onshift_dashers",
    "total_busy_dashers",
    "total_outstanding_orders",
)

def prepare_data(raw_df: pd.DataFrame) -> pd.DataFrame:
    df = raw_df.copy()
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
    df["actual_delivery_time"] = pd.to_datetime(df["actual_delivery_time"], errors="coerce")

    for col in ("market_id", "order_protocol"):
        df[col] = df[col].astype("Int64").astype("category")

    duration_sec = (df["actual_delivery_time"] - df["created_at"]).dt.total_seconds()
    df["delivery_duration_seconds"] = duration_sec
    df["delivery_duration_minutes"] = duration_sec / 60.0

    # Remover apenas alvos ausentes ou <= 0 (sem remoção de outliers)
    valid_mask = duration_sec.notna() & (duration_sec > 0)
    removed = int((~valid_mask).sum())
    df = df.loc[valid_mask].copy()
    print(f"Registros removidos (alvo ausente ou <= 0): {removed} | Restantes: {len(df)}")

    # Converter negativos em NaN apenas nas 4 colunas especificadas (sem imputação)
    for col in NEGATIVE_TO_NAN_COLS:
        neg_mask = df[col] < 0
        if neg_mask.any():
            print(f"  Negativos -> NaN em {col}: {int(neg_mask.sum())}")
            df.loc[neg_mask, col] = np.nan

    return df

df_model = prepare_data(df_raw)

## 3. Engenharia de Atributos e Controle de Vazamento (Etapas 04 e 05)

In [ ]:
def build_features(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()

    # Temporais (mantidas no horário registrado; provável UTC sem offset local)
    df["created_hour"] = df["created_at"].dt.hour
    df["created_dayofweek"] = df["created_at"].dt.dayofweek
    df["created_month"] = df["created_at"].dt.month
    df["created_is_weekend"] = (df["created_dayofweek"] >= 5).astype("int8")
    df["created_time_block"] = pd.cut(
        df["created_hour"],
        bins=[-1, 5, 11, 17, 23],
        labels=["00-05h_reg", "06-11h_reg", "12-17h_reg", "18-23h_reg"],
    )

    # Pedido
    valid_items = df["total_items"].notna() & (df["total_items"] > 0)
    df["average_item_price"] = np.where(
        valid_items & df["subtotal"].notna(),
        df["subtotal"] / df["total_items"],
        np.nan,
    )
    df["item_price_range"] = df["max_item_price"] - df["min_item_price"]
    df["invalid_item_price_range"] = (
        df["item_price_range"].notna() & (df["item_price_range"] < 0)
    ).astype("int8")
    df["distinct_item_ratio"] = np.where(
        valid_items & df["num_distinct_items"].notna(),
        df["num_distinct_items"] / df["total_items"],
        np.nan,
    )
    df["is_single_item"] = (df["total_items"] == 1).astype("int8")
    df["is_large_order"] = (df["total_items"] >= 5).astype("int8")

    # Operacionais experimentais (disponibilidade incerta; sem teto de 100% e sem imputar zero)
    both_valid = df["total_busy_dashers"].notna() & df["total_onshift_dashers"].notna()
    df["busy_greater_than_onshift"] = pd.Series(
        np.where(both_valid, df["total_busy_dashers"] > df["total_onshift_dashers"], pd.NA),
        index=df.index,
        dtype="boolean",
    )
    df["dasher_count_difference"] = df["total_onshift_dashers"] - df["total_busy_dashers"]
    df["is_operational_missing"] = (
        df[["total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders"]]
        .isna().any(axis=1).astype("int8")
    )

    valid_onshift = df["total_onshift_dashers"].notna() & (df["total_onshift_dashers"] > 0)
    df["outstanding_orders_per_onshift_dasher"] = np.where(
        valid_onshift & df["total_outstanding_orders"].notna(),
        df["total_outstanding_orders"] / df["total_onshift_dashers"],
        np.nan,
    )
    df["busy_dashers_ratio"] = np.where(
        valid_onshift & df["total_busy_dashers"].notna(),
        df["total_busy_dashers"] / df["total_onshift_dashers"],
        np.nan,
    )
    df["estimated_total_duration"] = (
        df["estimated_order_place_duration"] + df["estimated_store_to_consumer_driving_duration"]
    )

    # Contagem histórica estritamente anterior por loja (tratando empates de timestamp)
    orig_index = df.index
    ts_counts = (
        df.groupby(["store_id", "created_at"], sort=True)
        .size()
        .rename("_ts_count")
        .reset_index()
        .sort_values(["store_id", "created_at"], kind="mergesort")
    )
    ts_counts["historical_store_order_count"] = (
        ts_counts.groupby("store_id", sort=False)["_ts_count"].cumsum() - ts_counts["_ts_count"]
    )
    df = df.merge(
        ts_counts[["store_id", "created_at", "historical_store_order_count"]],
        on=["store_id", "created_at"],
        how="left",
    )
    df.index = orig_index
    return df

df_feat = build_features(df_model)

# Separação rigorosa de atributos
# Nota: store_id (6.743 IDs) é excluído do One-Hot Encoding nesta etapa para evitar
# explosão dimensional em memória nos modelos baseados em árvore.
CONFIRMED_MODEL_FEATURES = [
    "market_id", "store_primary_category", "order_protocol",
    "total_items", "subtotal", "num_distinct_items", "min_item_price", "max_item_price",
    "estimated_order_place_duration",
    "created_hour", "created_dayofweek", "created_month", "created_is_weekend", "created_time_block",
    "average_item_price", "item_price_range", "invalid_item_price_range",
    "distinct_item_ratio", "is_single_item", "is_large_order", "historical_store_order_count",
]

FORBIDDEN_LEAKAGE_COLS = [
    "actual_delivery_time", "delivery_duration_seconds", "delivery_duration_minutes",
]

print(f"Shape df_feat: {df_feat.shape} | Features confirmadas: {len(CONFIRMED_MODEL_FEATURES)}")

## 4. Validação Temporal com Embargo (Etapa 05)

In [ ]:
VAL_START = pd.Timestamp("2015-02-10 10:00:00")
TEST_START = pd.Timestamp("2015-02-14 10:00:00")

in_train = df_feat["created_at"] < VAL_START
in_val = (df_feat["created_at"] >= VAL_START) & (df_feat["created_at"] < TEST_START)
in_test = df_feat["created_at"] >= TEST_START

cross_tr_va = in_train & (df_feat["actual_delivery_time"] >= VAL_START)
cross_va_te = in_val & (df_feat["actual_delivery_time"] >= TEST_START)

train_df = df_feat.loc[in_train & ~cross_tr_va].copy()
val_df = df_feat.loc[in_val & ~cross_va_te].copy()
test_df = df_feat.loc[in_test].copy()  # Mantido completamente isolado

assert len(train_df) == 138572, f"Treino esperado 138572, obtido {len(train_df)}"
assert len(val_df) == 28808, f"Validação esperada 28808, obtido {len(val_df)}"
assert len(test_df) == 30041, f"Teste esperado 30041, obtido {len(test_df)}"
assert len(set(train_df.index) & set(val_df.index)) == 0
assert len(set(train_df.index) & set(test_df.index)) == 0
assert len(set(val_df.index) & set(test_df.index)) == 0
assert train_df["actual_delivery_time"].max() < VAL_START
assert val_df["actual_delivery_time"].max() < TEST_START

print(f"Treino: {len(train_df)} | Validação: {len(val_df)} | Teste (isolado): {len(test_df)}")

## 5. Pré-processamento, Baseline e Modelos Iniciais (Features Confirmadas)

In [ ]:
TARGET_COL = "delivery_duration_minutes"
CATEGORICAL_COLS = ["market_id", "store_primary_category", "order_protocol", "created_time_block"]
BOOLEAN_ABLATION_COLS = ["busy_greater_than_onshift"]

def prepare_feature_frame(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    for c in cols:
        if c in FORBIDDEN_LEAKAGE_COLS:
            raise ValueError(f"Coluna proibida detectada: {c}")
    out = df[cols].copy()
    for c in CATEGORICAL_COLS:
        if c in out.columns:
            out[c] = out[c].astype("object").where(out[c].notna(), np.nan).astype(str)
            out.loc[out[c].isin(["nan", "<NA>"]), c] = np.nan
    for c in BOOLEAN_ABLATION_COLS:
        if c in out.columns:
            out[c] = out[c].astype("float64")
    return out

def build_preprocessor(cols: list, scale_numeric: bool = False) -> ColumnTransformer:
    cat_cols = [c for c in CATEGORICAL_COLS if c in cols]
    num_cols = [c for c in cols if c not in cat_cols]
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", cat_pipe, cat_cols),
    ])

def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    abs_err = np.abs(y_true - y_pred)
    diff = y_pred - y_true
    return {
        "MAE": round(float(mean_absolute_error(y_true, y_pred)), 4),
        "RMSE": round(float(root_mean_squared_error(y_true, y_pred)), 4),
        "MedAE": round(float(median_absolute_error(y_true, y_pred)), 4),
        "P90_AE": round(float(np.percentile(abs_err, 90)), 4),
        "R2": round(float(r2_score(y_true, y_pred)), 4),
        "Mean_Bias": round(float(np.mean(diff)), 4),
        "Median_Bias": round(float(np.median(diff)), 4),
    }

X_train = prepare_feature_frame(train_df, CONFIRMED_MODEL_FEATURES)
y_train = train_df[TARGET_COL].values
X_val = prepare_feature_frame(val_df, CONFIRMED_MODEL_FEATURES)
y_val = val_df[TARGET_COL].values

models = {
    "Baseline (Median)": Pipeline([
        ("prep", build_preprocessor(CONFIRMED_MODEL_FEATURES, scale_numeric=False)),
        ("model", DummyRegressor(strategy="median")),
    ]),
    "Linear Regression": Pipeline([
        ("prep", build_preprocessor(CONFIRMED_MODEL_FEATURES, scale_numeric=True)),
        ("model", LinearRegression()),
    ]),
    "Random Forest": Pipeline([
        ("prep", build_preprocessor(CONFIRMED_MODEL_FEATURES, scale_numeric=False)),
        ("model", RandomForestRegressor(
            n_estimators=100, max_depth=15, min_samples_leaf=10,
            n_jobs=-1, random_state=RANDOM_STATE,
        )),
    ]),
    "HistGradientBoosting": Pipeline([
        ("prep", build_preprocessor(CONFIRMED_MODEL_FEATURES, scale_numeric=False)),
        ("model", HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.08, max_leaf_nodes=63,
            min_samples_leaf=30, random_state=RANDOM_STATE,
        )),
    ]),
}

results = []
preds = {}
for name, pipe in models.items():
    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    y_pred = pipe.predict(X_val)
    infer_time = time.perf_counter() - t1

    m = compute_metrics(y_val, y_pred)
    m["Model"] = name
    m["Train_Time_s"] = round(train_time, 2)
    m["Infer_Time_s"] = round(infer_time, 3)
    results.append(m)
    preds[name] = y_pred

results_df = pd.DataFrame(results).sort_values("MAE").reset_index(drop=True)
display(results_df[["Model", "MAE", "RMSE", "MedAE", "P90_AE", "R2", "Mean_Bias", "Median_Bias", "Train_Time_s", "Infer_Time_s"]])

## 6. Avaliação por Faixa de Duração Real e Gráficos Diagnósticos

In [ ]:
best_name = results_df.iloc[0]["Model"]
y_pred_best = preds[best_name]

bins = [0, 30, 45, 60, 90, 120, np.inf]
labels = ["<= 30 min", "30-45 min", "45-60 min", "60-90 min", "90-120 min", "> 120 min"]
band_eval = pd.DataFrame({
    "y_true": y_val,
    "y_pred": y_pred_best,
    "abs_err": np.abs(y_val - y_pred_best),
    "bias": y_pred_best - y_val,
    "band": pd.cut(y_val, bins=bins, labels=labels, right=True),
})
band_df = band_eval.groupby("band", observed=False).agg(
    n=("y_true", "count"),
    MAE=("abs_err", "mean"),
    MedAE=("abs_err", "median"),
    P90_AE=("abs_err", lambda s: np.percentile(s, 90)),
    Mean_Bias=("bias", "mean"),
).round(2)
print(f"Erro por faixa de duração real ({best_name}):")
display(band_df)

fig, axes = plt.subplots(2, 2, figsize=(14, 11))
mask_180 = y_val <= 180
axes[0, 0].scatter(y_val[mask_180], y_pred_best[mask_180], alpha=0.12, s=8, color="#2b5c8f")
axes[0, 0].plot([0, 180], [0, 180], "r--", linewidth=1.5)
axes[0, 0].set_title(f"Reais vs Previstos — {best_name} (<= 180 min)")
axes[0, 0].set_xlabel("Duração Real (min)")
axes[0, 0].set_ylabel("Duração Prevista (min)")

residuals = y_val - y_pred_best
sns.histplot(residuals[(residuals >= -60) & (residuals <= 120)], bins=70, kde=True, ax=axes[0, 1], color="#4c72b0")
axes[0, 1].axvline(0, color="red", linestyle="--")
axes[0, 1].set_title("Distribuição dos Resíduos (y_real - y_prev)")
axes[0, 1].set_xlabel("Resíduo (min)")

band_df["MAE"].plot(kind="bar", ax=axes[1, 0], color="#dd8452", label="MAE")
band_df["MedAE"].plot(kind="line", marker="o", ax=axes[1, 0], color="#2b5c8f", linewidth=2, label="MedAE")
axes[1, 0].set_title("Erro Absoluto por Faixa de Duração Real")
axes[1, 0].set_ylabel("Erro (min)")
axes[1, 0].tick_params(axis="x", rotation=20)
axes[1, 0].legend()

x = np.arange(len(results_df))
w = 0.35
axes[1, 1].bar(x - w/2, results_df["MAE"], width=w, label="MAE", color="#2b5c8f")
axes[1, 1].bar(x + w/2, results_df["RMSE"], width=w, label="RMSE", color="#c44e52")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(results_df["Model"], rotation=15)
axes[1, 1].set_title("Comparação MAE vs RMSE (Validação)")
axes[1, 1].set_ylabel("Erro (min)")
axes[1, 1].legend()

fig.tight_layout()
plt.show()

## 7. Experimentos de Ablação (EXPERIMENTAIS — Disponibilidade Incerta)

In [ ]:
ablation_sets = {
    "Conservador (Confirmadas)": CONFIRMED_MODEL_FEATURES,
    "Exp 1 (+ Estimativas Driving/Total)": CONFIRMED_MODEL_FEATURES + [
        "estimated_store_to_consumer_driving_duration", "estimated_total_duration",
    ],
    "Exp 2 (+ Operacionais Base)": CONFIRMED_MODEL_FEATURES + [
        "total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders",
        "dasher_count_difference", "busy_greater_than_onshift", "is_operational_missing",
    ],
    "Exp 3 (+ Razoes Operacionais)": CONFIRMED_MODEL_FEATURES + [
        "outstanding_orders_per_onshift_dasher", "busy_dashers_ratio",
    ],
    "Exp 4 (+ Todas Experimentais)": CONFIRMED_MODEL_FEATURES + [
        "estimated_store_to_consumer_driving_duration", "estimated_total_duration",
        "total_onshift_dashers", "total_busy_dashers", "total_outstanding_orders",
        "dasher_count_difference", "busy_greater_than_onshift", "is_operational_missing",
        "outstanding_orders_per_onshift_dasher", "busy_dashers_ratio",
    ],
}

ablation_rows = []
for exp_name, cols in ablation_sets.items():
    X_tr = prepare_feature_frame(train_df, cols)
    X_va = prepare_feature_frame(val_df, cols)
    pipe = Pipeline([
        ("prep", build_preprocessor(cols, scale_numeric=False)),
        ("model", HistGradientBoostingRegressor(
            max_iter=200, learning_rate=0.08, max_leaf_nodes=63,
            min_samples_leaf=30, random_state=RANDOM_STATE,
        )),
    ])
    t0 = time.perf_counter()
    pipe.fit(X_tr, y_train)
    train_time = time.perf_counter() - t0
    y_p = pipe.predict(X_va)
    m = compute_metrics(y_val, y_p)
    m["Experimento"] = exp_name
    m["N_Features"] = len(cols)
    m["Train_Time_s"] = round(train_time, 2)
    ablation_rows.append(m)

ablation_df = pd.DataFrame(ablation_rows).sort_values("MAE").reset_index(drop=True)
display(ablation_df[["Experimento", "MAE", "RMSE", "MedAE", "P90_AE", "R2", "Mean_Bias", "N_Features", "Train_Time_s"]])